# Traitement des données : Dans cette partie, il s'agira pour nous de préparer les données pour la modélisation à travers le nettoyage, l'encodage et le feature engineering

In [1]:
# Importating necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

In [2]:
# Charging the dataset
data = pd.read_csv("../data/raw/Bank-Customer-Churn-Prediction.csv")

In [3]:
# Types of columns in the dataset
data.dtypes

customer_id           int64
credit_score          int64
country              object
gender               object
age                   int64
tenure                int64
balance             float64
products_number       int64
credit_card           int64
active_member         int64
estimated_salary    float64
churn                 int64
dtype: object

In [4]:
# Checking missing values
data.isnull().sum()

customer_id         0
credit_score        0
country             0
gender              0
age                 0
tenure              0
balance             0
products_number     0
credit_card         0
active_member       0
estimated_salary    0
churn               0
dtype: int64

In [5]:
# Removing 'customer_id' column definitely which not useful for us
data = data.drop(columns=['customer_id'])

In [6]:
# Visualization of the first 3 rows of the dataset after removing the 'customer_id' column
data.head(3)

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1


In [7]:
# Checking for duplicates
print(data.duplicated().sum())

0


In [8]:
# Checking categorial columns and let's look at the unique values
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", categorical_cols)
for col in categorical_cols:
    print(f"\n{col}: {data[col].unique()}")

Categorical columns: ['country', 'gender']

country: ['France' 'Spain' 'Germany']

gender: ['Female' 'Male']


## Encodage des variables catégorielles

In [9]:
# Encoding categorical variables using OneHotEncoder
encoder = OneHotEncoder(drop='first', sparse_output=False)
encoded_features = encoder.fit_transform(data[categorical_cols])
encoded_feature_names = encoder.get_feature_names_out(categorical_cols)
data_encoded_features = pd.DataFrame(encoded_features, columns=encoded_feature_names)
data_encoded = data.drop(columns=categorical_cols)
data_encoded = pd.concat([data_encoded, data_encoded_features], axis=1)

In [10]:
# Let's verify all columns are numeric
data_encoded.dtypes

credit_score          int64
age                   int64
tenure                int64
balance             float64
products_number       int64
credit_card           int64
active_member         int64
estimated_salary    float64
churn                 int64
country_Germany     float64
country_Spain       float64
gender_Male         float64
dtype: object

In [11]:
# Displaying the first 3 rows of the encoded dataset
data_encoded.head(3)

,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,country_Germany,country_Spain,gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,0.0,0.0,0.0
1,608,41,1,83807.86,1,0,1,112542.58,0,0.0,1.0,0.0
2,502,42,8,159660.80,3,1,0,113931.57,1,0.0,0.0,0.0


## Feature Engineering
Sur la base des résultats de l'EDA ('01_eda.ipynb'), notamment les boxplots 
et la matrice de corrélation, nous allons créer de nouvelles variables afin 
de capturer les relations non linéaires que les variables originales 
ne suffisent pas à exprimer.

### Justification
Les corrélations linéaires observées restent globalement faibles (< 0.3), 
ce qui indique que le churn est un phénomène complexe nécessitant des 
features plus expressives.
Age : +0.285 | Élevée ---
Active_member : -0.156 | Élevée ---
Balance : +0.119 | Élevée ---
Tenure : -0.014 | Moyenne ---
Products_number : -0.048 | Moyenne 

### Nouvelles features à créer
- **age_group** : segmentation de l'âge en tranches pour capturer 
  les effets de seuil (< 30, 30-40, 40-50, 50-60, 60+)
- **has_balance** : indicateur binaire (balance > 0) pour isoler 
  les clients sans épargne active
- **tenure_group** : regroupement de l'ancienneté pour révéler 
  des schémas non linéaires (new, medium, loyal)
- **active_products** : interaction entre active_member et 
  products_number pour combiner deux signaux faibles en un 
  signal composite plus fort

Ces nouvelles variables visent à améliorer la capacité prédictive 
du modèle en rendant explicites les patterns détectés lors de l'EDA.

In [12]:
# Age groups
data_encoded['age_group'] = pd.cut(data_encoded['age'], 
                                 bins=[0, 30, 40, 50, 60, 100], 
                                 labels=['young', 'middle', 'senior', 'old', 'very_old'])
data_encoded = pd.get_dummies(data_encoded, columns=['age_group'], drop_first=True)

In [13]:
# Balance flag (has balance or not)
data_encoded['has_balance'] = (data_encoded['balance'] > 0).astype(int)

In [14]:
# Tenure groups
data_encoded['tenure_group'] = pd.cut(data_encoded['tenure'], 
                                    bins=[-1, 2, 5, 10],
                                    labels=['new', 'medium', 'loyal'])
data_encoded = pd.get_dummies(data_encoded, columns=['tenure_group'], drop_first=True)

In [15]:
# Active member interaction
data_encoded['active_products'] = data_encoded['active_member'] * data_encoded['products_number']

In [16]:
# Verification of the new features
data_encoded.shape
data_encoded.columns.tolist()


['credit_score',
 'age',
 'tenure',
 'balance',
 'products_number',
 'credit_card',
 'active_member',
 'estimated_salary',
 'churn',
 'country_Germany',
 'country_Spain',
 'gender_Male',
 'age_group_middle',
 'age_group_senior',
 'age_group_old',
 'age_group_very_old',
 'has_balance',
 'tenure_group_medium',
 'tenure_group_loyal',
 'active_products']

In [17]:
# Displaying the first 3 rows of the encoded dataset with new features
data_encoded.head(3)

,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,country_Germany,country_Spain,gender_Male,age_group_middle,age_group_senior,age_group_old,age_group_very_old,has_balance,tenure_group_medium,tenure_group_loyal,active_products
0,619,42,2,0.00,1,1,1,101348.88,1,0.0,0.0,0.0,False,True,False,False,0,False,False,1
1,608,41,1,83807.86,1,0,1,112542.58,0,0.0,1.0,0.0,False,True,False,False,1,False,False,1
2,502,42,8,159660.80,3,1,0,113931.57,1,0.0,0.0,0.0,False,True,False,False,1,False,True,0


In [18]:
# Correlation matrix for the encoded dataset with new features
correlation_matrix = data_encoded.corr()
correlation_matrix

,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,country_Germany,country_Spain,gender_Male,age_group_middle,age_group_senior,age_group_old,age_group_very_old,has_balance,tenure_group_medium,tenure_group_loyal,active_products
credit_score,1.000000,-0.003965,0.000842,0.006268,0.012238,-0.005458,0.025651,-0.001384,-0.027094,0.005538,0.004780,-0.002857,0.007991,-0.009863,-0.011471,0.009177,0.008380,0.002173,-0.004619,0.027226
age,-0.003965,1.000000,-0.009997,0.028308,-0.030680,-0.011721,0.085472,-0.007201,0.285323,0.046897,-0.001685,-0.027544,-0.287070,0.297314,0.449936,0.601328,0.034950,-0.004087,-0.011129,0.070612
tenure,0.000842,-0.009997,1.000000,-0.012254,0.013444,0.022583,-0.028362,0.007784,-0.014001,-0.000567,0.003868,0.014733,0.000002,-0.005300,-0.001558,-0.008209,-0.015235,-0.229582,0.863795,-0.016501
balance,0.006268,0.028308,-0.012254,1.000000,-0.304180,-0.014858,-0.010084,0.012797,0.118533,0.401110,-0.134892,0.012087,-0.012955,0.023223,0.027902,-0.002628,0.922780,-0.004364,-0.008586,-0.117188
products_number,0.012238,-0.030680,0.013444,-0.304180,1.000000,0.003183,0.009612,0.014204,-0.047820,-0.010419,0.009039,-0.021859,0.006255,-0.002063,-0.033373,-0.008184,-0.329294,0.005284,0.000443,0.340676
credit_card,-0.005458,-0.011721,0.022583,-0.014858,0.003183,1.000000,-0.011866,-0.009933,-0.007138,0.010577,-0.013480,0.005766,0.012722,-0.012869,-0.013191,0.000676,-0.018358,0.007387,0.016972,-0.006431
active_member,0.025651,0.085472,-0.028362,-0.010084,0.009612,-0.011866,1.000000,-0.011421,-0.156128,-0.020486,0.016732,0.022544,-0.031287,-0.047892,0.037284,0.129361,-0.004116,-0.008921,-0.016838,0.882012
estimated_salary,-0.001384,-0.007201,0.007784,0.012797,0.014204,-0.009933,-0.011421,1.000000,0.012097,0.010297,-0.006482,-0.008112,-0.011383,0.018702,-0.015041,-0.007534,0.014486,-0.005515,0.006932,-0.001283
churn,-0.027094,0.285323,-0.014001,0.118533,-0.047820,-0.007138,-0.156128,0.012097,1.000000,0.173488,-0.052667,-0.106512,-0.184190,0.185535,0.261884,0.024178,0.122357,0.006422,-0.015687,-0.137902
country_Germany,0.005538,0.046897,-0.000567,0.401110,-0.010419,0.010577,-0.020486,0.010297,0.173488,1.000000,-0.332084,-0.024628,-0.043979,0.052412,0.036651,-0.003748,0.435655,-0.001614,-0.006744,-0.015009


### Note sur la multicolinéarité

Certaines features créées présentent une forte corrélation avec 
leur variable originale (has_balance/balance : 0.92, 
active_products/active_member : 0.88, 
tenure_group_loyal/tenure : 0.86, 
age_group_very_old/age : 0.6).

Ce problème n'est pas critique ici car nous utiliserons par la suite les modèles tree-based (LightGBM, Random Forest) qui sont insensibles à la multicolinéarité par nature. Aucune suppression n'est donc nécessaire.

Pour ce qui est des autres modèles à savoir la régression logistique, le Naive Bayes, le Gradient Descent et le SVM ces variables seront été écartées afin d'éviter des coefficients instables.

In [19]:
# Save
data_encoded.to_csv("../data/processed/churn_dataset_clean.csv", index=False)
print("Saved!")

Saved!
